In [2]:
import pandas as pd
from plotnine import *
import os
import json
import numpy as np
from test_reconstruction import get_summary_df

colors_dict = {
"SCoNE":"#332288",
"MVBC":"#44AA99",
"RGWAS":"#88CCEE",  
"C-NMF":"#d45087",  
"C-CoNE":"#f95d6a",  
"G-NMF":"#ff7c43",  
"G-CoNE":"#ffa600",  
"HNMF":"#117733",  
"SCoNE(Fro)":"#1f77b4",
"CoNE":"#17becf",
"HNMF(res)":"#bc5090"
}

In [4]:
root_dir = '/home/jupyter/repos/SCoNE'
output_dir = f'{root_dir}/output'
os.makedirs(f'{output_dir}/figures',exist_ok=True)

In [3]:
# make graph for ari
def simple_graph(summary,variable_name,x_axis_name,y_axis_name,color_name,group,output_path,colors_dict=None, linetype=None, legend_too_long=False, jitter=0, shape_var=None):
    if linetype is None and shape_var is None:
        p = ggplot(summary, aes(variable_name, "mean", color=color_name, group=group))
    elif shape_var is None:
        p = ggplot(summary, aes(variable_name, "mean", color=color_name, linetype=linetype, group=group))
    else: 
        p = ggplot(summary, aes(variable_name, "mean", color=color_name, linetype=linetype, group=group, shape=shape_var))
    if jitter:
        pos = position_dodge(width=0.01)
    else:
        pos = position_dodge(width=0)

    p = (
    p
    + geom_point(size=2, position=pos)
    + geom_line(position=pos)
    + geom_errorbar(aes(ymin="ymin",ymax="ymax"),width=0.05)
    + labs(y=y_axis_name,x=x_axis_name,color="")
    + theme_minimal()
        + theme(panel_border=element_rect(color="black", fill=None, size=1),
            panel_grid_major=element_blank(),
        panel_grid_minor=element_blank(),
        figure_size=(4, 4),text=element_text(family='sans-serif',size=10),
        legend_position="top"
    )
    )
    if colors_dict is None:
        p = p + scale_color_brewer(type="qual", palette="Paired")
    else:
        p = p + scale_color_manual(values=colors_dict)
    if legend_too_long:
        p = p + guides(
            color=guide_legend(nrow=2),
            linetype=guide_legend(nrow=2)
        )
    p.save(f'{output_path}.pdf',dpi=300)


In [5]:
def make_ari_graph(experiment_name, param_list=None, xvar=None, linetype_var=None):
    df = pd.read_csv(f'{output_dir}/{experiment_name}_results.csv', index_col=0)
    df_train = df[df['split']=='train'].copy()
    if param_list is None:
        summary_df = get_summary_df(df_train,["run_name", "sim_id", "job_id",experiment_name],"ari")
    else:
        summary_df = get_summary_df(df_train,["run_name", "sim_id", "job_id"]+param_list,"ari")
    if xvar is None: xvar=experiment_name

    if linetype_var is None:
        simple_graph(summary=summary_df,variable_name=xvar,
                     x_axis_name=xvar,y_axis_name="ari",color_name="run_name",group="run_name",
                     output_path=f'{output_dir}/figures/{experiment_name}_ari', colors_dict=colors_dict)
    else:
        summary_df['group'] = summary_df["run_name"].astype(str) + "_" + summary_df[linetype_var].astype(str)
        simple_graph(summary=summary_df,variable_name=xvar,
                     x_axis_name=xvar,y_axis_name="ari",color_name="run_name", linetype=linetype_var, 
                     legend_too_long=True,group="run_name",jitter=True,
                     output_path=f'{output_dir}/figures/{experiment_name}_ari', colors_dict=colors_dict)

In [6]:
def make_ccc_graph(experiment_name, param_list=None, xvar=None, linetype_var=None):
    df = pd.read_csv(f'{output_dir}/{experiment_name}_results.csv', index_col=0)
    df_train = df[df['split']=='train'].copy()
    if xvar is None: xvar=experiment_name
    if param_list is None:
        df_train = df_train[~df_train['coph_corr'].isna()].drop_duplicates(["run_name", "sim_id", "job_id", xvar])
        summary_df = get_summary_df(df_train,["run_name", "sim_id", "job_id",experiment_name],"coph_corr")
    else:
        df_train = df_train[~df_train['coph_corr'].isna()].drop_duplicates(["run_name", "sim_id", "job_id"]+param_list)
        summary_df = get_summary_df(df_train,["run_name", "sim_id", "job_id"]+param_list,"coph_corr")
    
    if linetype_var is None:
        simple_graph(summary=summary_df,variable_name=xvar,
                     x_axis_name=xvar,y_axis_name="coph_corr",color_name="run_name",group="run_name",
                     output_path=f'{output_dir}/figures/{experiment_name}_cophcorr')
    else:
        summary_df['group'] = summary_df["run_name"].astype(str) + "_" + summary_df[linetype_var].astype(str)
        simple_graph(summary=summary_df,variable_name=xvar,
                     x_axis_name=xvar,y_axis_name="coph_corr",color_name="run_name", linetype=linetype_var, 
                     legend_too_long=True,group="group",
                     output_path=f'{output_dir}/figures/{experiment_name}_cophcorr')

In [7]:
def make_sim_graph(experiment_name, param_list=None, xvar=None, shape_var=None, run_name='SCoNE'):
    if xvar is None: xvar=experiment_name
    if param_list is None: param_list = [xvar]
    df = pd.read_csv(f'{output_dir}/{experiment_name}_results.csv', index_col=0)
    df_train = df[(df['split']=='train')&(df['run_name']==run_name)].copy()
    fm_columns = [i for i in df_train.columns if i.startswith('sim_') 
                and not '_null' in i and i!='sim_id']
    fm_results = []
    for fm_column in fm_columns:
        summary_df = get_summary_df(df_train,["run_name", "sim_id", "job_id"]+param_list,fm_column)
        summary_df['factor_matrix'] = fm_column.replace('sim_','')
        summary_df['null'] = False
        fm_results.append(summary_df)
        summary_df_null = get_summary_df(df_train,["run_name", "sim_id", "job_id"]+param_list,f'{fm_column}_null')
        summary_df_null['factor_matrix'] = fm_column.replace('sim_','')
        summary_df_null['null'] = True
        fm_results.append(summary_df_null)
    fm_results = pd.concat(fm_results)
    if shape_var is None:
        fm_results["group"] = fm_results["factor_matrix"].astype(str) + "_" + fm_results["null"].astype(str)
    else:
        fm_results["group"] = fm_results["factor_matrix"].astype(str) + "_" + fm_results["null"].astype(str) + "_" + fm_results[shape_var].astype(str)
    simple_graph(summary=fm_results,variable_name=xvar,x_axis_name=xvar,y_axis_name="Similarity",
             color_name="factor_matrix",output_path=f'{output_dir}/figures/{experiment_name}_sim_{run_name}', linetype="null", group="group", legend_too_long=True, shape_var=shape_var)

# Assess runs over rG

In [8]:
make_ari_graph("rG")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rG_ari.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 10 rows containing missing values.


In [10]:
make_sim_graph("rG")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rG_sim_SCoNE.pdf


In [9]:
make_ccc_graph("rG")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rG_cophcorr.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 30 rows containing missing values.


# Assess runs over rGC

In [70]:
make_ari_graph("rGC")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rGC_ari.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 10 rows containing missing values.


In [98]:
make_sim_graph("rGC")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rGC_sim.pdf


In [99]:
make_ccc_graph("rGC")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rGC_cophcorr.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 30 rows containing missing values.


# Assess runs over rZ

In [15]:
make_ari_graph(experiment_name="rZ_unobs", param_list=["rZ", "unobs_conf"], xvar="rZ", linetype_var="unobs_conf")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rZ_unobs_ari.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 10 rows containing missing values.


In [21]:
df = pd.read_csv(f'{output_dir}/rZ_unobs_results.csv', index_col=0)
df_train = df[(df['split']=='train')].copy()
df_train.drop_duplicates(['run_name','unobs_conf'])

,ari,sim_W,sim_W_null,tuning_metric,run,split,sim_id,job_id,sim_H_G,sim_H_G_null,...,sim_U_C,sim_U_C_null,rel_error_G,rel_error_C,coph_corr,run_name,lambda_option,rZ,unobs_conf,out_path
5050,0.004331,0.286605,0.242405,-0.004331,0,train,5,45,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,MVBC,1.000,0.3,False,/home/jupyter/repos/SCoNE/output/models/rZ_uno...
5054,0.023198,0.421750,0.360329,0.163955,0,train,5,40,0.370225,0.272840,...,NaN,NaN,0.050087,0.113868,0.939430,HNMF(res),0.000,0.3,False,/home/jupyter/repos/SCoNE/output/models/rZ_uno...
5255,0.384086,0.714661,0.427620,1.057707,0,train,7,56,0.309726,0.321257,...,NaN,NaN,0.437397,0.620310,0.942070,HNMF,0.000,0.4,False,/home/jupyter/repos/SCoNE/output/models/rZ_uno...
5456,0.066071,0.526068,0.421652,1.126545,0,train,2,17,0.331772,0.292348,...,0.272560,0.135114,0.450173,0.676373,0.960749,SCoNE,0.000,0.2,True,/home/jupyter/repos/SCoNE/output/models/rZ_uno...
5757,0.392396,0.712051,0.428061,1.057841,0,train,5,47,0.310522,0.283935,...,0.287663,0.132724,0.441889,0.615952,0.988081,SCoNE,0.001,0.3,False,/home/jupyter/repos/SCoNE/output/models/rZ_uno...


In [ ]:
make_ari_graph(experiment_name="rZ_unobs", param_list=["rZ", "unobs_conf"], xvar="rZ", linetype_var="unobs_conf")

In [29]:
make_sim_graph(experiment_name="rZ_unobs", xvar="rZ", param_list=["rZ", "unobs_conf"],shape_var="unobs_conf", run_name='SCoNE')

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rZ_unobs_sim_SCoNE.pdf


In [22]:
make_sim_graph(experiment_name="rZ_unobs", xvar="rZ")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rZ_unobs_sim.pdf


In [35]:
xvar="rZ"
experiment_name="rZ_unobs"
param_list=["rZ","unobs_conf"]

if xvar is None: xvar=experiment_name
if param_list is None: param_list = [xvar]
df = pd.read_csv(f'{output_dir}/{experiment_name}_results.csv', index_col=0)
df_train = df[(df['split']=='train')].copy()
fm_columns = [i for i in df_train.columns if i.startswith('sim_') 
            and not '_null' in i and i!='sim_id']
fm_results = []
for fm_column in fm_columns:
    summary_df = get_summary_df(df_train,["run_name", "sim_id", "job_id"]+param_list,fm_column)
    summary_df['factor_matrix'] = fm_column.replace('sim_','')
    summary_df['null'] = False
    fm_results.append(summary_df)
    summary_df_null = get_summary_df(df_train,["run_name", "sim_id", "job_id"]+param_list,f'{fm_column}_null')
    summary_df_null['factor_matrix'] = fm_column.replace('sim_','')
    summary_df_null['null'] = True
    fm_results.append(summary_df_null)
fm_results = pd.concat(fm_results)


In [37]:
fm_results

,run_name,sim_id,job_id,rZ,unobs_conf,n,mean,std,se,ymin,ymax,factor_matrix,null
0,HNMF,1,5,0.1,False,50,0.645503,0.024745,0.003500,0.642003,0.649002,W,False
1,HNMF,3,22,0.2,False,50,0.668930,0.042246,0.005975,0.662956,0.674905,W,False
2,HNMF,5,39,0.3,False,50,0.693441,0.038597,0.005458,0.687982,0.698899,W,False
3,HNMF,7,56,0.4,False,50,0.691949,0.067885,0.009600,0.682349,0.701550,W,False
4,HNMF,9,73,0.5,False,50,0.715427,0.065784,0.009303,0.706123,0.724730,W,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
45,SCoNE,15,132,0.8,False,50,0.131383,0.011922,0.001686,0.129697,0.133068,U_C,True
46,SCoNE,16,139,0.9,True,50,0.138884,0.008088,0.001144,0.137740,0.140028,U_C,True
47,SCoNE,17,151,0.9,False,50,0.140932,0.008865,0.001254,0.139678,0.142185,U_C,True
48,SCoNE,18,155,1.0,True,50,0.137946,0.008641,0.001222,0.136724,0.139168,U_C,True


In [43]:
fm_results["group"] = fm_results["run_name"].astype(str) + "_" + fm_results["null"].astype(str) + "_" + fm_results["unobs_conf"].astype(str)
simple_graph(fm_results[fm_results['factor_matrix']=='H_G'],
             variable_name="rZ",x_axis_name="rZ",y_axis_name="Similarity",
             color_name="run_name",group="group",output_path=f'{output_dir}/H_G',colors_dict=colors_dict, linetype="null", legend_too_long=True, jitter=0.01, shape_var="unobs_conf")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/H_G.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_point : Removed 20 rows containing missing values.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 20 rows containing missing values.


# Assess runs over signed effects

In [44]:
make_ari_graph("signed_cov_effects")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/signed_cov_effects_ari.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 2 rows containing missing values.


In [47]:
make_sim_graph(experiment_name="signed_cov_effects")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/signed_cov_effects_sim_SCoNE.pdf


In [48]:
make_ccc_graph(experiment_name="signed_cov_effects")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/signed_cov_effects_cophcorr.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 6 rows containing missing values.


# Assess runs over overdispersion

In [49]:
make_ari_graph("overdispersion_nu")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/overdispersion_nu_ari.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 4 rows containing missing values.


In [51]:
make_sim_graph("overdispersion_nu")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/overdispersion_nu_sim_SCoNE.pdf


In [50]:
make_ccc_graph("overdispersion_nu")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/overdispersion_nu_cophcorr.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 12 rows containing missing values.


# Assess runs over rank misspecification

In [57]:
make_ccc_graph("rank_subgroupstructure", param_list=["rank","subgroup_structure"], xvar="rank", linetype_var="subgroup_structure")

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rank_subgroupstructure_cophcorr.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 24 rows containing missing values.


In [60]:
make_ari_graph("rank_subgroupstructure",xvar="rank", param_list=["rank", "subgroup_structure"])

/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 4 x 4 in image.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: /home/jupyter/repos/SCoNE/output/figures/rank_subgroupstructure_ari.pdf
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_point : Removed 12 rows containing missing values.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/geoms/geom_path.py:100: PlotnineWarning: geom_path: Removed 1 rows containing missing values.
/opt/conda/envs/py311/lib/python3.14/site-packages/plotnine/layer.py:374: PlotnineWarning: geom_errorbar : Removed 16 rows containing missing values.
